In [1]:
import os
import numpy as np

import qkeras
from qkeras import QActivation
from qkeras import QConv1D
from qkeras.quantizers import quantized_bits, quantized_relu, quantized_tanh

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, UpSampling1D, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger



resolution = 150
X_total = np.load(f"Data/X_Data_Bank_{resolution}.npy")
Y_total = np.load(f"Data/Y_Data_Bank_{resolution}.npy")
X_data = X_total[:1500,:,:]
y_data = Y_total[:1500,:,:]
TIME_STEPS = np.shape(X_data)[1]

VAL_SPLIT = 0.1

SAVE_DIR_opt = '1Conv_checkpoints_running_opt'
LOG_FILE_opt = '1Conv_3pulse_noise_tb23_opt.csv'
MODEL_NAME_TEMPLATE_opt = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv_opt.h5'
checkpoint_path_opt = os.path.join(SAVE_DIR_opt, MODEL_NAME_TEMPLATE_opt)


wq = quantized_bits(8, 2, alpha=1) 
aq = quantized_relu(6) 

qmodel = Sequential([
    Input(shape=(TIME_STEPS, 1)),

    QConv1D(32, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),

    QConv1D(32, 3, strides=2, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),

    QConv1D(64, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),

    UpSampling1D(2),

    QConv1D(32, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),
    
    QConv1D(1, 1, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    
    QActivation("quantized_sigmoid")
    ])

qmodel.summary()


optimizer = Adam(learning_rate=0.001)
qmodel.compile(loss='binary_crossentropy', optimizer=optimizer) 


if not os.path.isdir(SAVE_DIR_opt):
    os.makedirs(SAVE_DIR_opt)

callbacks = [
    ModelCheckpoint(checkpoint_path_opt, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    CSVLogger(LOG_FILE_opt, append=True, separator=','),
    EarlyStopping(monitor="val_loss", mode="min", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1) 
]


qmodel.fit(
    X_data, y_data,
    epochs=150,                  
    shuffle=True,
    validation_split=VAL_SPLIT,
    callbacks=callbacks
)

if not os.path.isdir("hgdream_new"):
    os.makedirs("hgdream_new")
    
qmodel.save(f"hgdream_new/quantized_model_opt_{resolution}.h5")

2026-05-07 15:53:22.968417: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-07 15:53:24.943477: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv1d (QConv1D)          (None, 80, 32)            128       
                                                                 
 q_activation (QActivation)  (None, 80, 32)            0         
                                                                 
 q_conv1d_1 (QConv1D)        (None, 40, 32)            3104      
                                                                 
 q_activation_1 (QActivation  (None, 40, 32)           0         
 )                                                               
                                                                 
 q_conv1d_2 (QConv1D)        (None, 40, 64)          

Epoch 22/150
36/43 [========================>.....] - ETA: 0s - loss: 0.1148
Epoch 22: val_loss improved from 0.11972 to 0.11923, saving model to 1Conv_checkpoints_running_opt/1Conv_2pulse_noise.loss_0.11923.e022_deconv_opt.h5
43/43 [==============================] - 0s 8ms/step - loss: 0.1143 - val_loss: 0.1192 - lr: 2.5000e-04
Epoch 23/150
42/43 [============================>.] - ETA: 0s - loss: 0.1134
Epoch 23: val_loss did not improve from 0.11923
43/43 [==============================] - 0s 11ms/step - loss: 0.1135 - val_loss: 0.1222 - lr: 2.5000e-04
Epoch 24/150
42/43 [============================>.] - ETA: 0s - loss: 0.1136
Epoch 24: val_loss did not improve from 0.11923
43/43 [==============================] - 1s 12ms/step - loss: 0.1137 - val_loss: 0.1249 - lr: 2.5000e-04
Epoch 25/150
37/43 [========================>.....] - ETA: 0s - loss: 0.1124
Epoch 25: val_loss did not improve from 0.11923
43/43 [==============================] - 0s 8ms/step - loss: 0.1131 - val_loss: 0.12

In [2]:
import hls4ml
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)
print(co)
hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity='name',
    backend='Vitis'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='model_final',
    part='xcu250-figd2104-2L-e',
    io_type='io_stream',
)
hls_model.compile()

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-daqavh9u because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
/opt/conda/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


{'QInitializer': <class 'qkeras.qlayers.QInitializer'>, 'QDense': <class 'qkeras.qlayers.QDense'>, 'QConv1D': <class 'qkeras.qconvolutional.QConv1D'>, 'QConv2D': <class 'qkeras.qconvolutional.QConv2D'>, 'QConv2DTranspose': <class 'qkeras.qconvolutional.QConv2DTranspose'>, 'QSimpleRNNCell': <class 'qkeras.qrecurrent.QSimpleRNNCell'>, 'QSimpleRNN': <class 'qkeras.qrecurrent.QSimpleRNN'>, 'QLSTMCell': <class 'qkeras.qrecurrent.QLSTMCell'>, 'QLSTM': <class 'qkeras.qrecurrent.QLSTM'>, 'QGRUCell': <class 'qkeras.qrecurrent.QGRUCell'>, 'QGRU': <class 'qkeras.qrecurrent.QGRU'>, 'QBidirectional': <class 'qkeras.qrecurrent.QBidirectional'>, 'QDepthwiseConv2D': <class 'qkeras.qconvolutional.QDepthwiseConv2D'>, 'QSeparableConv1D': <class 'qkeras.qconvolutional.QSeparableConv1D'>, 'QSeparableConv2D': <class 'qkeras.qconvolutional.QSeparableConv2D'>, 'QActivation': <class 'qkeras.qlayers.QActivation'>, 'QAdaptiveActivation': <class 'qkeras.qlayers.QAdaptiveActivation'>, 'QBatchNormalization': <class